# Notebook 03: Dataset Preparation Basics

## 📚 Historical Context

**Timeline**: Evolution of Data Management in ML (2010-2024)

**The Problem (Pre-2015)**:
- **Custom scripts**: Every project reinvented data loading
- **RAM limitations**: Load entire dataset into memory → crashes
- **Slow pandas**: Inefficient for large text datasets
- **Format chaos**: CSV, JSON, XML, custom formats everywhere
- **No standardization**: Each model needed different format

**The Evolution**:
- **2015**: TensorFlow TFRecord format for efficient storage
- **2016**: PyTorch DataLoader with lazy loading
- **2019**: HuggingFace Datasets library (Apache Arrow backend)
- **2020**: Dataset streaming for massive datasets
- **2022**: Datasets Hub - sharable, versioned datasets
- **2023**: Multi-modal datasets (text + images + audio)

**Modern Reality**:
- **HuggingFace Datasets**: 10,000+ datasets, one interface
- **Streaming**: Train on 1TB+ datasets without download
- **Efficient**: Apache Arrow → 10-100x faster than pandas
- **Shareable**: Push dataset to Hub, others can use instantly

## 🎯 What You'll Learn

1. **Loading from Various Formats** - CSV, JSON, JSONL, Parquet, HF datasets
2. **Dataset Formats for Tasks** - Classification, generation, instruction-following
3. **Train/Val/Test Splits** - Random, stratified, time-based
4. **Dataset Streaming** - Handle datasets larger than RAM
5. **Preprocessing Pipelines** - Tokenization, filtering, mapping
6. **Saving & Loading** - Efficient storage and reload
7. **Data Validation** - Catch errors before training

## 💡 Why This Matters

**Bad data preparation = Bad model, guaranteed:**
- **Wrong format** = Training crashes or wrong loss
- **Data leakage** = Overly optimistic results, fails in production
- **Inefficient loading** = 90% of training time wasted on I/O
- **No validation** = Discover issues after 24 hours of training

**Real Example**:
```python
# Inefficient (loads all to RAM, slow)
df = pd.read_csv('large_file.csv')  # 50GB file → OOM crash

# Efficient (streaming, fast)
dataset = load_dataset('csv', data_files='large_file.csv', streaming=True)
```

---

In [ ]:
# Import libraries
import os
import json
import csv
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Union, Optional
from collections import Counter

# HuggingFace
from datasets import (
    load_dataset,
    Dataset,
    DatasetDict,
    Features,
    Value,
    ClassLabel,
)
from transformers import AutoTokenizer

# Set random seed
np.random.seed(42)

# Plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Create data directory if it doesn't exist
data_dir = Path.cwd().parent / 'data'
data_dir.mkdir(exist_ok=True)
(data_dir / 'raw').mkdir(exist_ok=True)
(data_dir / 'processed').mkdir(exist_ok=True)
(data_dir / 'samples').mkdir(exist_ok=True)

print("Libraries imported successfully!")
print(f"Data directory: {data_dir}")

## Part 1: Loading Data from Different Formats

### Common Data Formats:

| Format | Use Case | Pros | Cons |
|--------|----------|------|------|
| **CSV** | Tabular, spreadsheets | Human-readable, Excel compatible | Large files slow, encoding issues |
| **JSON** | Nested structures | Flexible schema | Large file size |
| **JSONL** | Streaming, large datasets | Line-by-line reading | Not human-friendly for large files |
| **Parquet** | Big data, analytics | Compressed, columnar, fast | Binary (need tools to view) |
| **HF Dataset** | ML datasets | Optimized for ML, shareable | Requires datasets library |

### When to Use What:
- **CSV**: Small-medium datasets, coming from Excel/spreadsheets
- **JSONL**: Large text datasets, streaming needed
- **Parquet**: Very large datasets, need fast columnar access
- **HF Dataset**: Sharing datasets, need reproducibility

In [ ]:
# Create sample datasets in different formats
print("=" * 60)
print("CREATING SAMPLE DATASETS")
print("=" * 60)

# Sample data - sentiment analysis dataset
sample_data = [
    {"text": "I love this product! It's amazing.", "label": "positive", "rating": 5},
    {"text": "Terrible experience, waste of money.", "label": "negative", "rating": 1},
    {"text": "It's okay, nothing special.", "label": "neutral", "rating": 3},
    {"text": "Absolutely fantastic! Highly recommend.", "label": "positive", "rating": 5},
    {"text": "Poor quality, broke after one day.", "label": "negative", "rating": 1},
    {"text": "Average product, met expectations.", "label": "neutral", "rating": 3},
    {"text": "Best purchase I've ever made!", "label": "positive", "rating": 5},
    {"text": "Not worth the price at all.", "label": "negative", "rating": 2},
]

# 1. Save as CSV
csv_path = data_dir / 'samples' / 'sentiment_data.csv'
df = pd.DataFrame(sample_data)
df.to_csv(csv_path, index=False)
print(f"✓ Created CSV: {csv_path}")

# 2. Save as JSON
json_path = data_dir / 'samples' / 'sentiment_data.json'
with open(json_path, 'w') as f:
    json.dump(sample_data, f, indent=2)
print(f"✓ Created JSON: {json_path}")

# 3. Save as JSONL (JSON Lines - one JSON per line)
jsonl_path = data_dir / 'samples' / 'sentiment_data.jsonl'
with open(jsonl_path, 'w') as f:
    for item in sample_data:
        f.write(json.dumps(item) + '\n')
print(f"✓ Created JSONL: {jsonl_path}")

# 4. Save as Parquet
parquet_path = data_dir / 'samples' / 'sentiment_data.parquet'
df.to_parquet(parquet_path, index=False)
print(f"✓ Created Parquet: {parquet_path}")

# Show file sizes
print("\n" + "=" * 60)
print("FILE SIZE COMPARISON")
print("=" * 60)
for path in [csv_path, json_path, jsonl_path, parquet_path]:
    size = path.stat().st_size
    print(f"{path.suffix:8s}: {size:6d} bytes")

print("\nNote: Parquet is usually smallest for large datasets (compressed)")
print("=" * 60)

In [ ]:
# Load data from different formats
print("=" * 60)
print("LOADING DATA FROM DIFFERENT FORMATS")
print("=" * 60)

# Method 1: Load CSV with pandas
print("\n1. Loading CSV with pandas:")
df_csv = pd.read_csv(csv_path)
print(f"   Shape: {df_csv.shape}")
print(f"   Columns: {df_csv.columns.tolist()}")
print(f"   First row: {df_csv.iloc[0].to_dict()}")

# Method 2: Load CSV with HuggingFace datasets (better for large files)
print("\n2. Loading CSV with HuggingFace datasets:")
dataset_csv = load_dataset('csv', data_files=str(csv_path), split='train')
print(f"   Length: {len(dataset_csv)}")
print(f"   Features: {dataset_csv.features}")
print(f"   First item: {dataset_csv[0]}")

# Method 3: Load JSON
print("\n3. Loading JSON with HuggingFace datasets:")
dataset_json = load_dataset('json', data_files=str(json_path), split='train')
print(f"   Length: {len(dataset_json)}")
print(f"   First item: {dataset_json[0]}")

# Method 4: Load JSONL
print("\n4. Loading JSONL:")
dataset_jsonl = load_dataset('json', data_files=str(jsonl_path), split='train')
print(f"   Length: {len(dataset_jsonl)}")
print(f"   First item: {dataset_jsonl[0]}")

# Method 5: Load Parquet
print("\n5. Loading Parquet:")
dataset_parquet = load_dataset('parquet', data_files=str(parquet_path), split='train')
print(f"   Length: {len(dataset_parquet)}")
print(f"   First item: {dataset_parquet[0]}")

# Method 6: Load from HuggingFace Hub (public datasets)
print("\n6. Loading from HuggingFace Hub (example: IMDB):")
try:
    # Load a small sample
    dataset_hub = load_dataset('imdb', split='train[:5]')  # First 5 examples
    print(f"   Length: {len(dataset_hub)}")
    print(f"   Features: {dataset_hub.features}")
    print(f"   First text (truncated): {dataset_hub[0]['text'][:100]}...")
except Exception as e:
    print(f"   Note: Requires internet. Error: {e}")

print("\n" + "=" * 60)
print("KEY TAKEAWAYS")
print("=" * 60)
print("1. Use HuggingFace datasets.load_dataset() for consistency")
print("2. It handles CSV, JSON, JSONL, Parquet with same interface")
print("3. Returns Dataset object optimized for ML workflows")
print("4. Much faster than pandas for large files (Apache Arrow backend)")
print("=" * 60)

## Part 2: Dataset Formats for Different Tasks

### Classification Format:
```json
{"text": "...", "label": 0}  // or "positive", "negative"
```

### Text Generation Format:
```json
{"text": "Once upon a time..."}  // Just text, model predicts next tokens
```

### Instruction Following Format (Most Common for Fine-Tuning):
```json
{
  "instruction": "Translate to French:",
  "input": "Hello, how are you?",
  "output": "Bonjour, comment allez-vous?"
}
```

### Chat Format (Multi-turn):
```json
{
  "messages": [
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"}
  ]
}
```

### Question Answering Format:
```json
{
  "context": "Paris is the capital of France...",
  "question": "What is the capital of France?",
  "answer": "Paris"
}
```

In [ ]:
print("=" * 60)
print("DATASET FORMATS FOR DIFFERENT TASKS")
print("=" * 60)

# 1. Classification Dataset
print("\n1. CLASSIFICATION FORMAT:")
print("-" * 40)
classification_data = [
    {"text": "This movie is great!", "label": "positive"},
    {"text": "Terrible film, waste of time.", "label": "negative"},
]
print(json.dumps(classification_data[0], indent=2))
print("\nUse case: Sentiment analysis, topic classification, spam detection")

# 2. Text Generation Dataset
print("\n2. TEXT GENERATION FORMAT:")
print("-" * 40)
generation_data = [
    {"text": "Once upon a time, there was a brave knight who"},
    {"text": "The future of artificial intelligence will be"},
]
print(json.dumps(generation_data[0], indent=2))
print("\nUse case: Story continuation, code completion, autocompletion")

# 3. Instruction Following Dataset
print("\n3. INSTRUCTION FOLLOWING FORMAT:")
print("-" * 40)
instruction_data = [
    {
        "instruction": "Translate the following English text to French:",
        "input": "Hello, how are you?",
        "output": "Bonjour, comment allez-vous?"
    },
    {
        "instruction": "Summarize the following text:",
        "input": "Long article text here...",
        "output": "Brief summary here..."
    },
]
print(json.dumps(instruction_data[0], indent=2))
print("\nUse case: Instruction-tuned models (like GPT-3.5, Llama-2-Chat)")
print("Most versatile format for modern LLM fine-tuning")

# 4. Chat Format
print("\n4. CHAT FORMAT (Multi-turn):")
print("-" * 40)
chat_data = [
    {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is Python?"},
            {"role": "assistant", "content": "Python is a programming language..."},
            {"role": "user", "content": "Can you show me an example?"},
            {"role": "assistant", "content": "Sure! Here's a simple example: print('Hello')"}
        ]
    }
]
print(json.dumps(chat_data[0], indent=2))
print("\nUse case: Chatbots, conversational AI, multi-turn dialogue")

# 5. Question Answering Format
print("\n5. QUESTION ANSWERING FORMAT:")
print("-" * 40)
qa_data = [
    {
        "context": "Paris is the capital and largest city of France. It is located in the north-central part of the country.",
        "question": "What is the capital of France?",
        "answer": "Paris"
    }
]
print(json.dumps(qa_data[0], indent=2))
print("\nUse case: Reading comprehension, extractive QA")

print("\n" + "=" * 60)
print("CHOOSING THE RIGHT FORMAT")
print("=" * 60)
print("Classification:        → Labels are fixed, predicting category")
print("Generation:            → Open-ended text output")
print("Instruction Following: → Task + Input + Expected Output")
print("Chat:                  → Multi-turn conversations")
print("QA:                    → Context + Question → Extract answer")
print("\nMost versatile: Instruction Following format")
print("Can represent almost any task!")
print("=" * 60)

In [ ]:
# Convert between formats (example)
print("=" * 60)
print("CONVERTING FORMATS")
print("=" * 60)

# Original: Classification format
classification_example = {
    "text": "This product is amazing!",
    "label": "positive"
}

print("\nOriginal (Classification format):")
print(json.dumps(classification_example, indent=2))

# Convert to: Instruction Following format
instruction_example = {
    "instruction": "Classify the sentiment of the following text as positive, negative, or neutral:",
    "input": classification_example["text"],
    "output": classification_example["label"]
}

print("\nConverted to (Instruction Following format):")
print(json.dumps(instruction_example, indent=2))

print("\n" + "=" * 60)
print("WHY INSTRUCTION FORMAT IS POPULAR")
print("=" * 60)
print("1. Can represent ANY task (classification, generation, QA, etc.)")
print("2. More explicit about what the model should do")
print("3. Enables few-shot learning (examples in instruction)")
print("4. Used by most modern fine-tuning approaches (Alpaca, Vicuna, etc.)")
print("5. Same format for all tasks → one training pipeline")
print("=" * 60)

## Part 3: Train/Validation/Test Splitting Strategies

### Why Split?
- **Train**: Model learns from this data
- **Validation**: Tune hyperparameters, early stopping
- **Test**: Final evaluation (never seen during training)

### Common Splits:
- **80/10/10**: Large datasets
- **70/15/15**: Medium datasets
- **60/20/20**: Small datasets (need more validation data)

### Splitting Strategies:

**1. Random Split**
- Shuffle and split randomly
- Good for: IID (independent, identically distributed) data
- Bad for: Time series, grouped data

**2. Stratified Split**
- Maintain class distribution in all splits
- Good for: Imbalanced classification
- Critical for: Rare classes

**3. Time-based Split**
- Older data → train, recent → test
- Good for: Time series, temporal data
- Simulates real deployment

**4. Group-based Split**
- Keep related items together
- Good for: User data, multi-turn conversations
- Prevents leakage

In [ ]:
from sklearn.model_selection import train_test_split

print("=" * 60)
print("TRAIN/VAL/TEST SPLITTING STRATEGIES")
print("=" * 60)

# Create a larger sample dataset
np.random.seed(42)
n_samples = 1000

# Imbalanced dataset (more positive than negative)
labels = ['positive'] * 500 + ['negative'] * 300 + ['neutral'] * 200
texts = [f"Sample text {i}" for i in range(n_samples)]
indices = list(range(n_samples))

# Shuffle
combined = list(zip(texts, labels, indices))
np.random.shuffle(combined)
texts, labels, indices = zip(*combined)

# Convert to lists
texts = list(texts)
labels = list(labels)
indices = list(indices)

print(f"\nTotal samples: {len(texts)}")
print(f"Label distribution: {Counter(labels)}")

# Strategy 1: Random Split
print("\n" + "=" * 60)
print("1. RANDOM SPLIT (80/10/10)")
print("=" * 60)

# First split: 80% train, 20% temp
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# Second split: split temp into val and test (50/50 of temp = 10/10 of total)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

print(f"\nTrain: {len(train_texts)} samples ({len(train_texts)/len(texts)*100:.1f}%)")
print(f"Val:   {len(val_texts)} samples ({len(val_texts)/len(texts)*100:.1f}%)")
print(f"Test:  {len(test_texts)} samples ({len(test_texts)/len(texts)*100:.1f}%)")

print(f"\nTrain distribution: {Counter(train_labels)}")
print(f"Val distribution:   {Counter(val_labels)}")
print(f"Test distribution:  {Counter(test_labels)}")

print("\n⚠️  Notice: Distributions are similar but not exactly the same")

# Strategy 2: Stratified Split
print("\n" + "=" * 60)
print("2. STRATIFIED SPLIT (80/10/10)")
print("=" * 60)

# First split: 80% train, 20% temp (stratified)
train_texts_s, temp_texts_s, train_labels_s, temp_labels_s = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# Second split: split temp into val and test (stratified)
val_texts_s, test_texts_s, val_labels_s, test_labels_s = train_test_split(
    temp_texts_s, temp_labels_s, test_size=0.5, random_state=42, stratify=temp_labels_s
)

print(f"\nTrain: {len(train_texts_s)} samples ({len(train_texts_s)/len(texts)*100:.1f}%)")
print(f"Val:   {len(val_texts_s)} samples ({len(val_texts_s)/len(texts)*100:.1f}%)")
print(f"Test:  {len(test_texts_s)} samples ({len(test_texts_s)/len(texts)*100:.1f}%)")

print(f"\nTrain distribution: {Counter(train_labels_s)}")
print(f"Val distribution:   {Counter(val_labels_s)}")
print(f"Test distribution:  {Counter(test_labels_s)}")

print("\n✓ Notice: Distributions are now proportionally identical!")

# Visualize the difference
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Random split
for i, (split_labels, title) in enumerate([
    (train_labels, 'Train'),
    (val_labels, 'Val'),
    (test_labels, 'Test')
]):
    counts = Counter(split_labels)
    axes[0, i].bar(counts.keys(), counts.values(), alpha=0.7, color='steelblue')
    axes[0, i].set_title(f'Random Split - {title}')
    axes[0, i].set_ylabel('Count')
    axes[0, i].grid(axis='y', alpha=0.3)

# Stratified split
for i, (split_labels, title) in enumerate([
    (train_labels_s, 'Train'),
    (val_labels_s, 'Val'),
    (test_labels_s, 'Test')
]):
    counts = Counter(split_labels)
    axes[1, i].bar(counts.keys(), counts.values(), alpha=0.7, color='coral')
    axes[1, i].set_title(f'Stratified Split - {title}')
    axes[1, i].set_ylabel('Count')
    axes[1, i].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("WHEN TO USE STRATIFIED SPLITTING")
print("=" * 60)
print("✓ Classification tasks with imbalanced classes")
print("✓ Small datasets (preserve class distribution)")
print("✓ Rare classes (ensure they appear in all splits)")
print("✗ NOT for time series (breaks temporal order)")
print("✗ NOT for generation tasks (no classes)")
print("=" * 60)

In [ ]:
# Using HuggingFace datasets for splitting
print("=" * 60)
print("SPLITTING WITH HUGGINGFACE DATASETS")
print("=" * 60)

# Create a Dataset
data_dict = {
    'text': texts,
    'label': labels
}
dataset = Dataset.from_dict(data_dict)

print(f"\nOriginal dataset: {len(dataset)} samples")

# Method 1: Simple train_test_split
print("\nMethod 1: Simple split (80/20)")
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
print(f"  Train: {len(split_dataset['train'])} samples")
print(f"  Test:  {len(split_dataset['test'])} samples")

# Method 2: Three-way split
print("\nMethod 2: Three-way split (80/10/10)")
# First split: train vs temp
train_test = dataset.train_test_split(test_size=0.2, seed=42)
# Second split: val vs test
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

# Create final dataset dict
final_dataset = DatasetDict({
    'train': train_test['train'],
    'validation': val_test['train'],
    'test': val_test['test']
})

print(f"  Train:      {len(final_dataset['train'])} samples")
print(f"  Validation: {len(final_dataset['validation'])} samples")
print(f"  Test:       {len(final_dataset['test'])} samples")

# Method 3: Stratified split (if needed)
print("\nMethod 3: Stratified split")
print("  Note: HuggingFace datasets doesn't have built-in stratification")
print("  Solution: Use sklearn first, then create Dataset")

# Save the split dataset
output_dir = data_dir / 'processed' / 'sentiment_split'
final_dataset.save_to_disk(str(output_dir))
print(f"\n✓ Saved split dataset to: {output_dir}")

# Load it back
loaded_dataset = DatasetDict.load_from_disk(str(output_dir))
print(f"✓ Loaded dataset: {loaded_dataset}")

print("\n" + "=" * 60)

## Part 4: Dataset Streaming for Large Files

### The Problem:
```python
# This will crash if file is larger than RAM
dataset = load_dataset('json', data_files='huge_file.jsonl')  # 100GB
```

### The Solution: Streaming
```python
# This works even for 1TB+ files!
dataset = load_dataset('json', data_files='huge_file.jsonl', streaming=True)
```

### How It Works:
- **No full load**: Reads data line-by-line
- **Iterator**: Access via iteration, not indexing
- **Memory efficient**: Constant memory usage
- **Trade-off**: Can't use all Dataset operations

### When to Use:
- ✓ Dataset larger than RAM
- ✓ Exploring data before download
- ✓ Training with very large corpora
- ✗ Need random access by index
- ✗ Need to shuffle (requires loading)
- ✗ Need to sort or filter heavily

In [ ]:
print("=" * 60)
print("DATASET STREAMING")
print("=" * 60)

# Create a larger sample file for demonstration
large_file = data_dir / 'samples' / 'large_data.jsonl'
print(f"\nCreating sample large file: {large_file}")

with open(large_file, 'w') as f:
    for i in range(10000):  # 10k samples
        item = {
            'id': i,
            'text': f'This is sample text number {i}. ' * 10,  # Make it longer
            'label': np.random.choice(['positive', 'negative', 'neutral'])
        }
        f.write(json.dumps(item) + '\n')

file_size = large_file.stat().st_size / (1024 * 1024)  # MB
print(f"Created file size: {file_size:.2f} MB")

# Method 1: Regular loading (loads entire file)
print("\n" + "=" * 60)
print("METHOD 1: REGULAR LOADING")
print("=" * 60)
print("Loading entire dataset into memory...")

dataset_regular = load_dataset('json', data_files=str(large_file), split='train')
print(f"✓ Loaded: {len(dataset_regular)} samples")
print(f"✓ Can access by index: dataset[0] = {dataset_regular[0]['id']}")
print(f"✓ Can shuffle, filter, sort easily")
print(f"✗ Uses RAM: ~{file_size:.2f} MB")

# Method 2: Streaming (processes line by line)
print("\n" + "=" * 60)
print("METHOD 2: STREAMING")
print("=" * 60)
print("Streaming dataset (line by line)...")

dataset_streaming = load_dataset('json', data_files=str(large_file), split='train', streaming=True)
print(f"✓ Dataset ready (not loaded yet)")
print(f"✓ Memory usage: constant (very low)")
print(f"✗ Can't access by index (no dataset[0])")
print(f"✓ Can iterate: for item in dataset")

# Iterate through first few items
print("\nIterating through first 5 items:")
for i, item in enumerate(dataset_streaming):
    if i >= 5:
        break
    print(f"  Item {i}: ID={item['id']}, Label={item['label']}")

# Streaming with .map() for preprocessing
print("\n" + "=" * 60)
print("STREAMING WITH PREPROCESSING")
print("=" * 60)

def preprocess(example):
    """Simple preprocessing function."""
    example['text_length'] = len(example['text'])
    example['text_upper'] = example['text'].upper()
    return example

# Apply preprocessing while streaming
dataset_preprocessed = dataset_streaming.map(preprocess)

print("Preprocessing applied (lazy evaluation)")
print("Processing happens during iteration:\n")

for i, item in enumerate(dataset_preprocessed):
    if i >= 3:
        break
    print(f"  Item {i}:")
    print(f"    Text length: {item['text_length']}")
    print(f"    First 50 chars: {item['text_upper'][:50]}...")
    print()

print("=" * 60)
print("STREAMING: PROS AND CONS")
print("=" * 60)
print("✓ PROS:")
print("  - Works with datasets larger than RAM")
print("  - Constant memory usage")
print("  - Can start processing immediately")
print("  - Can apply .map() for preprocessing")
print("\n✗ CONS:")
print("  - No random access by index")
print("  - No len() (don't know total count)")
print("  - Shuffling is limited")
print("  - Some operations not available")
print("\n💡 WHEN TO USE:")
print("  - Dataset > 50% of available RAM")
print("  - Training on very large corpora")
print("  - Exploring data before full download")
print("=" * 60)

## Part 5: Building Preprocessing Pipelines

### Common Preprocessing Steps:

**1. Text Cleaning**
- Remove HTML tags
- Fix encoding issues
- Normalize whitespace

**2. Tokenization**
- Convert text to token IDs
- Add special tokens
- Create attention masks

**3. Filtering**
- Remove too short/long sequences
- Remove low-quality samples
- Remove duplicates

**4. Formatting**
- Convert to model-specific format
- Add task-specific fields

### Pipeline Pattern:
```python
dataset = (
    load_dataset(...)
    .map(clean_text)      # Step 1
    .filter(remove_short)  # Step 2
    .map(tokenize)        # Step 3
    .filter(filter_length) # Step 4
)
```

In [ ]:
import re
import html

print("=" * 60)
print("BUILDING PREPROCESSING PIPELINE")
print("=" * 60)

# Create messy sample data (realistic problems)
messy_data = [
    {"text": "<p>This is HTML text</p>  with   extra spaces.", "label": "positive"},
    {"text": "This has &amp; HTML entities &lt;tags&gt;.", "label": "negative"},
    {"text": "x", "label": "positive"},  # Too short
    {"text": "Normal text here.", "label": "neutral"},
    {"text": "SHOUTING TEXT!!!", "label": "negative"},
    {"text": "Text " + "word " * 200, "label": "positive"},  # Too long
    {"text": "   Leading and trailing spaces   ", "label": "neutral"},
    {"text": "Normal text.", "label": "positive"},
]

# Create dataset
dataset = Dataset.from_dict({
    'text': [item['text'] for item in messy_data],
    'label': [item['label'] for item in messy_data]
})

print(f"\nOriginal dataset: {len(dataset)} samples\n")
print("Sample data (first 3):")
for i in range(min(3, len(dataset))):
    print(f"  {i}: '{dataset[i]['text'][:60]}...'")

# Step 1: Clean text
print("\n" + "=" * 60)
print("STEP 1: CLEAN TEXT")
print("=" * 60)

def clean_text(example):
    """Clean text: remove HTML, decode entities, normalize whitespace."""
    text = example['text']
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # Decode HTML entities
    text = html.unescape(text)
    
    # Normalize whitespace (multiple spaces → single space)
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    example['text_cleaned'] = text
    return example

dataset = dataset.map(clean_text)
print("\nAfter cleaning (first 3):")
for i in range(min(3, len(dataset))):
    print(f"  {i}: '{dataset[i]['text_cleaned'][:60]}...'")

# Step 2: Filter by length
print("\n" + "=" * 60)
print("STEP 2: FILTER BY LENGTH")
print("=" * 60)

def is_valid_length(example, min_length=10, max_length=500):
    """Keep only samples within length range."""
    length = len(example['text_cleaned'])
    return min_length <= length <= max_length

print(f"Before filtering: {len(dataset)} samples")
dataset = dataset.filter(is_valid_length)
print(f"After filtering: {len(dataset)} samples")
print(f"Removed: {len(messy_data) - len(dataset)} samples (too short/long)")

# Step 3: Tokenize
print("\n" + "=" * 60)
print("STEP 3: TOKENIZE")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    """
    Tokenize texts.
    
    Note: This operates on batches for efficiency.
    """
    return tokenizer(
        examples['text_cleaned'],
        truncation=True,
        max_length=128,
        padding='max_length',
        return_tensors=None  # Return lists, not tensors
    )

# Apply tokenization in batches (much faster!)
dataset_tokenized = dataset.map(
    tokenize_function,
    batched=True,  # Process in batches for speed
    batch_size=4,
    remove_columns=['text', 'text_cleaned']  # Remove text, keep only IDs
)

print(f"\nTokenized dataset: {len(dataset_tokenized)} samples")
print(f"Features: {dataset_tokenized.features}")
print(f"\nSample tokenized data:")
print(f"  Input IDs: {dataset_tokenized[0]['input_ids'][:20]}... (first 20)")
print(f"  Attention mask: {dataset_tokenized[0]['attention_mask'][:20]}... (first 20)")

# Step 4: Convert labels to integers
print("\n" + "=" * 60)
print("STEP 4: CONVERT LABELS")
print("=" * 60)

# Create label mapping
label_list = ['positive', 'negative', 'neutral']
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(f"Label mapping: {label2id}")

def convert_labels(example):
    """Convert string labels to integers."""
    example['label'] = label2id[example['label']]
    return example

dataset_final = dataset_tokenized.map(convert_labels)

print(f"\nFinal dataset: {len(dataset_final)} samples")
print(f"Features: {dataset_final.features}")
print(f"\nSample final data:")
print(f"  Label: {dataset_final[0]['label']} ({id2label[dataset_final[0]['label']]})")
print(f"  Input IDs shape: {len(dataset_final[0]['input_ids'])}")

print("\n" + "=" * 60)
print("COMPLETE PIPELINE SUMMARY")
print("=" * 60)
print(f"1. Load data:        {len(messy_data)} samples")
print(f"2. Clean text:       {len(messy_data)} samples")
print(f"3. Filter length:    {len(dataset)} samples")
print(f"4. Tokenize:         {len(dataset_tokenized)} samples")
print(f"5. Convert labels:   {len(dataset_final)} samples")
print(f"\nFinal: {len(dataset_final)} samples ready for training")
print("=" * 60)

## Part 6: Saving and Loading Processed Datasets

### Why Save Processed Data?
- **Avoid reprocessing**: Tokenization can take hours on large datasets
- **Reproducibility**: Same preprocessing every time
- **Sharing**: Others can use your processed data

### Formats:

**1. HuggingFace Dataset format (.arrow files)**
- Fast loading (memory-mapped)
- Preserves all features and types
- Best for local use

**2. Push to Hub**
- Sharable via HuggingFace Hub
- Versioned
- Best for collaboration

**3. PyTorch DataLoader format**
- Convert to torch.utils.data.Dataset
- For training loop

In [ ]:
print("=" * 60)
print("SAVING AND LOADING DATASETS")
print("=" * 60)

# Save to disk (Arrow format)
print("\n1. SAVE TO DISK (Arrow format):")
print("-" * 40)

output_path = data_dir / 'processed' / 'sentiment_processed'
dataset_final.save_to_disk(str(output_path))
print(f"✓ Saved to: {output_path}")

# Check saved files
saved_files = list(output_path.glob('*'))
print(f"\nSaved files:")
for f in saved_files:
    size = f.stat().st_size
    print(f"  {f.name}: {size} bytes")

# Load from disk
print("\n2. LOAD FROM DISK:")
print("-" * 40)

loaded_dataset = Dataset.load_from_disk(str(output_path))
print(f"✓ Loaded: {len(loaded_dataset)} samples")
print(f"✓ Features: {loaded_dataset.features}")
print(f"✓ Sample: {loaded_dataset[0].keys()}")

# Save as CSV (for inspection)
print("\n3. SAVE AS CSV (for inspection):")
print("-" * 40)

csv_output = data_dir / 'processed' / 'sentiment_processed.csv'
# Note: Can't save tokenized data as CSV (has lists)
# Save original dataset instead
dataset.to_csv(str(csv_output))
print(f"✓ Saved to: {csv_output}")

# Save as JSON
print("\n4. SAVE AS JSON:")
print("-" * 40)

json_output = data_dir / 'processed' / 'sentiment_processed.json'
dataset.to_json(str(json_output))
print(f"✓ Saved to: {json_output}")

# Convert to PyTorch format
print("\n5. CONVERT TO PYTORCH FORMAT:")
print("-" * 40)

dataset_torch = dataset_final.with_format('torch')
print(f"✓ Converted to PyTorch format")
print(f"✓ Sample type: {type(dataset_torch[0]['input_ids'])}")
print(f"✓ Sample shape: {dataset_torch[0]['input_ids'].shape}")

print("\n" + "=" * 60)
print("BEST PRACTICES")
print("=" * 60)
print("1. Save processed datasets to avoid reprocessing")
print("2. Use .save_to_disk() for Arrow format (fastest)")
print("3. Save as CSV/JSON for inspection")
print("4. Use .with_format('torch') before training")
print("5. Version your datasets (add date/version to filename)")
print("=" * 60)

## 📊 Summary and Key Takeaways

### What We Learned:

**1. Loading Data**
- Multiple formats: CSV, JSON, JSONL, Parquet
- Use `load_dataset()` for consistency
- 10-100x faster than pandas for large files

**2. Dataset Formats**
- Classification: text + label
- Generation: just text
- Instruction: instruction + input + output (most versatile)
- Chat: multi-turn messages
- QA: context + question + answer

**3. Splitting Strategies**
- Random: shuffle and split
- Stratified: maintain class distribution (use for imbalanced data)
- Time-based: for temporal data
- Group-based: prevent leakage

**4. Streaming**
- For datasets larger than RAM
- Constant memory usage
- Trade-off: no random access

**5. Preprocessing Pipeline**
- Clean → Filter → Tokenize → Convert
- Use `.map()` for transformations
- Use `.filter()` for removing samples
- Process in batches for speed (`batched=True`)

**6. Saving/Loading**
- Save processed data to avoid reprocessing
- Arrow format: fastest
- CSV/JSON: for inspection
- `.with_format('torch')`: for training

### Critical for Fine-Tuning:

**Before Training Checklist:**
```python
# 1. Load data
dataset = load_dataset(...)

# 2. Clean and validate
dataset = dataset.map(clean_text)
dataset = dataset.filter(is_valid)

# 3. Split (stratified if classification)
dataset = dataset.train_test_split(test_size=0.2, stratify_by_column='label')

# 4. Tokenize
dataset = dataset.map(tokenize, batched=True)

# 5. Format for training
dataset = dataset.with_format('torch')

# 6. Save
dataset.save_to_disk('processed_dataset')
```

### Common Mistakes to Avoid:

| Mistake | Impact | Solution |
|---------|--------|----------|
| Loading with pandas | Slow, OOM | Use `load_dataset()` |
| Not splitting | Can't evaluate | Always split train/val/test |
| Random split on imbalanced | Poor validation | Use stratified split |
| Not filtering | Garbage in = garbage out | Filter aggressively |
| Tokenizing one-by-one | Very slow | Use `batched=True` |
| Not saving processed | Reprocess every time | Save with `.save_to_disk()` |

### Next Notebook:
**04_data_quality_analysis.ipynb**

We'll explore:
- Exploratory Data Analysis (EDA) for text
- Duplicate detection (exact and near-duplicates)
- Outlier detection
- Label consistency checking
- Text quality metrics
- Detecting toxic/harmful content
- PII detection and removal
- Data leakage detection

---

**Historical Note**: In the early days of NLP (pre-2015), every researcher wrote custom data loading code. This led to countless bugs, irreproducible results, and wasted time. The standardization brought by HuggingFace Datasets (2019) was revolutionary - suddenly, loading IMDB sentiment data required the same code as loading Wikipedia. This democratized NLP research and made it possible for anyone to work with large-scale datasets. Today, you can train on trillion-token datasets without writing a single line of custom data loading code.